# Step 1 — Document Ingestion & Cleaning
Loads all raw policy documents, cleans and segments them into clauses, and saves a unified `clauses_raw.csv` for downstream processing.

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath('../'))

import pandas as pd
from pathlib import Path

from src.ingestion.loader import load_all_documents
from src.ingestion.text_cleaner import clean_all_documents
from src.ingestion.document_segmentor import segment_document_into_clauses

RAW_DATA_PATH  = Path('../app/data/raw')
OUTPUT_PATH    = Path('../app/outputs/risk_reports')
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
print('Paths ready.')

Paths ready.


In [4]:
# Load all documents from data/raw
print('Loading documents...')
docs = load_all_documents(str(RAW_DATA_PATH))
print(f'Loaded {len(docs)} documents:')
for name, text in docs.items():
    print(f'  {name}: {len(text):,} characters')

Loading documents...
Loaded 3 documents:
  data_protection_policy.txt: 5,010 characters
  environmental_compliance.txt: 3,918 characters
  iaea_safety_standards.txt: 6,314 characters


In [5]:
# Clean all documents
print('Cleaning documents...')
cleaned_docs = clean_all_documents(docs)
print('Done.')

Cleaning documents...
Done.


In [6]:
# Segment into clauses
all_clauses = []
for doc_name, text in cleaned_docs.items():
    clauses = segment_document_into_clauses(text, doc_name=doc_name)
    for c in clauses:
        c['document'] = doc_name
    all_clauses.extend(clauses)
    print(f'{doc_name}: {len(clauses)} clauses')

print(f'\nTotal clauses: {len(all_clauses)}')

df = pd.DataFrame(all_clauses)
df['word_count'] = df['clause_text'].str.split().str.len()
df.head()

data_protection_policy.txt: 34 clauses
environmental_compliance.txt: 30 clauses
iaea_safety_standards.txt: 48 clauses

Total clauses: 112


,clause_id,section,clause_text,source,document,word_count
0,data_protection_policy.txt_0000,7.1 Designation,REGULATORY DOCUMENT: DATA PROTECTION AND PRIVA...,data_protection_policy.txt,data_protection_policy.txt,34
1,data_protection_policy.txt_0001,7.1 Designation,All entities subject to this regulation must c...,data_protection_policy.txt,data_protection_policy.txt,17
2,data_protection_policy.txt_0002,7.1 Designation,"""Personal data"" means any information relating...",data_protection_policy.txt,data_protection_policy.txt,13
3,data_protection_policy.txt_0003,7.1 Designation,"""Data controller"" refers to the entity that de...",data_protection_policy.txt,data_protection_policy.txt,16
4,data_protection_policy.txt_0004,7.1 Designation,"""Processing"" means any operation performed upo...",data_protection_policy.txt,data_protection_policy.txt,8


In [7]:
# Save raw clauses
out_file = OUTPUT_PATH / 'clauses_raw.csv'
df.to_csv(out_file, index=False)
print(f'Saved: {out_file}')
df.info()

Saved: ..\app\outputs\risk_reports\clauses_raw.csv
<class 'pandas.DataFrame'>
RangeIndex: 112 entries, 0 to 111
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   clause_id    112 non-null    str  
 1   section      112 non-null    str  
 2   clause_text  112 non-null    str  
 3   source       112 non-null    str  
 4   document     112 non-null    str  
 5   word_count   112 non-null    int64
dtypes: int64(1), str(5)
memory usage: 30.1 KB
